# Assignment 5: RAG Agent

**Data source:** Wikipedia — *Transformer (machine learning model)*

We build:
1. A **RAG Chain** — one retrieval + one LLM call
2. A **RAG Agent** — the LLM decides when and how many times to search

## Step 1 — Install dependencies

In [1]:
%pip install -q langchain langchain-core langchain-openai langchain-text-splitters langchain-community langchain-huggingface langgraph sentence-transformers bs4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Step 2 — Set up the model, embeddings, and vector store

In [2]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

llm = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

vector_store = InMemoryVectorStore(embeddings)

print("All components ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

All components ready!


## Step 3 — Load, split, and index the document

In [3]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Different data source from the lesson
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Transformer_(machine_learning_model)")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = splitter.split_documents(docs)

vector_store.add_documents(splits)
print(f"Indexed {len(splits)} chunks from the Wikipedia article")

Indexed 141 chunks from the Wikipedia article


## Step 4 — RAG Chain (simple, one call)

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

def rag_chain(query):
    docs = vector_store.similarity_search(query, k=3)
    context = "\n\n".join(doc.page_content for doc in docs)

    response = llm.invoke([
        SystemMessage(f"Answer using this context:\n\n{context}"),
        HumanMessage(query)
    ])
    response.pretty_print()

rag_chain("What is the attention mechanism in transformers?")

================================== Ai Message ==================================

**Attention in Transformers**

In a transformer, the **attention mechanism** is what lets each token look at (i.e., “pay attention to”) the other tokens in the sequence and decide how much information from each of those tokens should be mixed into its own representation.  

- **How it works** – For every token the model creates three learned vectors: a **query** ( \(W^{Q}\) ), a **key** ( \(W^{K}\) ), and a **value** ( \(W^{V}\) ).  
- **Scaled dot‑product attention** – The query vector of a token is dotted with the key vectors of all tokens, scaled by \(\sqrt{d_{\text{emb}}}\), to produce attention scores. These scores are turned into weights (via a soft‑max) and used to take a weighted sum of the value vectors.  
- **Multi‑head** – The same operation is performed in parallel in several “heads.” Each head learns a different kind of relationship (e.g., “next‑word” vs. “verb‑to‑object”), and the head outpu

## Step 5 — RAG Agent (multi-step, uses a tool)

In [5]:
from typing import Literal, Tuple, List
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool(response_format="content_and_artifact")
def retrieve(query: str, section: Literal["beginning", "middle", "end"] = "middle") -> Tuple[str, List[Document]]:
    """Search the Wikipedia article on Transformers to help answer a question."""
    docs = vector_store.similarity_search(query, k=3)
    content = "\n\n".join(doc.page_content for doc in docs)
    return content, docs

agent = create_react_agent(
    model=llm,
    tools=[retrieve],
    prompt="You are a helpful assistant. Use the retrieve tool to answer questions about the Transformer architecture."
)

print("Agent ready!")

Agent ready!


/tmp/ipykernel_899/131256578.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## Step 6 — Test the agent with a multi-step question

In [6]:
query = (
    "What problem did the Transformer architecture solve?\n\n"
    "Once you know that, look up what models were built using the Transformer."
)

result = agent.invoke({"messages": [HumanMessage(query)]})

for msg in result["messages"]:
    msg.pretty_print()
    print()

================================ Human Message =================================

What problem did the Transformer architecture solve?

Once you know that, look up what models were built using the Transformer.

================================== Ai Message ==================================
Tool Calls:
  retrieve (call_dd377e813d6c4b10ae0f4947)
 Call ID: call_dd377e813d6c4b10ae0f4947
  Args:
    section: beginning
    query: What problem did the Transformer architecture solve?

================================= Tool Message =================================
Name: retrieve

The modern version of the transformer was proposed in the 2017 paper "Attention Is All You Need" by researchers at Google.[1] The predecessors of transformers were developed as an improvement over previous architectures for machine translation,[4][5] but have found many applications since. They are used in large-scale natural language processing, computer vision (vision transformers), reinforcement learning,[6][7] au